In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
 
pd.set_option("future.no_silent_downcasting", True)
 
from laparoscopy_helpers.data_cleaning import to_snake_case, clean_surgical_df
from poor_patient_helpers.data_cleaning import load_all_endo, load_all_mercy, load_all_safe, build_final

In [2]:
path_corrected_names = '../Nkhoma_data/poor_patients_funds_data_corrected_names'

In [3]:
os.listdir(path_corrected_names)

['combined_clean.xlsx',
 'combined_clean_renamed.xlsx',
 'not_in_TB_colored.xlsx']

In [4]:
import pandas as pd

df1 = pd.read_excel(f"{path_corrected_names}/combined_clean.xlsx")
df2 = pd.read_excel(f"{path_corrected_names}/combined_clean_renamed.xlsx")

In [5]:
df1['_pos'] = df1.groupby('theatre_book_index').cumcount()

df2_clean = df2.drop_duplicates(subset=['theatre_book_index', 'first_name', 'last_name'], keep='first').copy()
df2_clean['_pos'] = df2_clean.groupby('theatre_book_index').cumcount()  # ← missing line

df1_corrected = df1.merge(
    df2_clean[['theatre_book_index', '_pos', 'first_name', 'last_name']],
    on=['theatre_book_index', '_pos'],
    how='left',
    suffixes=('_old', '_corrected')
)

# Show differences
diff_mask = (df1_corrected['first_name_old'] != df1_corrected['first_name_corrected']) | \
            (df1_corrected['last_name_old'] != df1_corrected['last_name_corrected'])
print(f"Name differences: {diff_mask.sum()}")
print(df1_corrected[diff_mask][['theatre_book_index', 'first_name_old', 'last_name_old',
                                 'first_name_corrected', 'last_name_corrected']].head(20))

# Apply corrections to df1
df1['first_name'] = df1_corrected['first_name_corrected'].fillna(df1_corrected['first_name_old'])
df1['last_name'] = df1_corrected['last_name_corrected'].fillna(df1_corrected['last_name_old'])
df1 = df1.drop(columns='_pos')

Name differences: 1813
      theatre_book_index first_name_old last_name_old first_name_corrected  \
77                220078            NaN           NaN                  NaN   
144               220145            NaN           NaN                  NaN   
217               220218            NaN           NaN                  NaN   
382               220383           LUMA        TALIFA                LUNIA   
554               220555         WONIZA          BISE             WELUZANI   
645               220646       LAWRENCE        KANTHU             LAWRENCE   
842               220843         HASSAN         SAIDI              HUSSEIN   
845               220846         NIMROD        PARASU              NIMRODI   
913               220914           MARY        EVANCE                 MARY   
1042              221043     LATE ENTRY           NaN           LATE ENTRY   
1268              221270     CHANONGOCH      CHILOMBO         CHIONONGANJI   
1428              221430         ERESTO  

In [6]:
df1

,theatre_book_index,hospital_id,date_of_surgery,first_name,last_name,age_years,sex,village,surgeon,1st_assistent_instructor,...,histology,finishing_time,urgency,surgery_severity,asascore,year_of_birth,operation_time_minutes,starting_time,continous_tb,year
0,220001,NaN,2022-01-01,ELIFA,SUMATI,26.0,F,Nkhonde,Obs/Gyn,NaN,...,No,NaN,NaN,NaN,NaN,1997.0,NaN,NaN,NaN,2022
1,220002,NaN,2022-01-01,SIYATU,ISAAC,27.0,F,Mozambique,Obs/Gyn,NaN,...,No,NaN,NaN,NaN,NaN,1996.0,NaN,NaN,NaN,2022
2,220003,NaN,2022-01-02,LONESS,MAPEMPHERO,25.0,F,Chembe,Obs/Gyn,NaN,...,No,NaN,NaN,NaN,NaN,1998.0,NaN,NaN,NaN,2022
3,220004,NaN,2022-01-03,SAIZI,NEDSON,48.0,M,Chilikumanda,Limbe,Caleb,...,No,NaN,Emergency,Major,ASA 3,1975.0,NaN,NaN,NaN,2022
4,220005,NaN,2022-01-03,BEATRICE,HEZEKIA,26.0,F,Mazengera,Obs/Gyn,NaN,...,No,NaN,NaN,NaN,NaN,1997.0,NaN,NaN,NaN,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7400,250644,NaN,2025-05-20,LAWRENCE,CHUNGA,53.0,M,Ntcheu,Widmann,Roma,...,NaN,NaN,Elective,NaN,NaN,NaN,NaN,NaN,251601.0,2025
7401,250839,NaN,2025-07-02,WILSON,KANYEMBA,58.0,M,Kachere,Widmann,NaN,...,No,NaN,Elective,NaN,ASA 1,NaN,NaN,NaN,251602.0,2025
7402,250899,19048.0,2025-07-15,LEVISON,UNDERSON,58.0,M,Mchezi,Lam,Madalitso,...,No,NaN,Elective,Minor,NaN,NaN,NaN,NaN,251603.0,2025
7403,251242,NaN,2025-10-02,ARGENT,LULANGA,62.0,M,Lilongwe,Limbe,Jonathan,...,No,NaN,Elective,Minor,ASA 2,NaN,NaN,NaN,251604.0,2025


In [7]:
df1.to_excel(f"{path_corrected_names}/combined_clean_corrected.xlsx", index=False)